In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pickle
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error, r2_score

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import KFold, RepeatedKFold, LeaveOneOut, GridSearchCV
from sklearn.model_selection import RandomizedSearchCV, cross_val_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

import warnings

In [ ]:
from tqdm.auto import tqdm
from sklearn import metrics
from sklearn.preprocessing import scale
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

from scipy.stats import randint, uniform

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
df_train = pd.read_parquet('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/Data/processed/df_train.parquet')
df_valid = pd.read_parquet('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/Data/processed/df_valid.parquet')
df_test = pd.read_parquet('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/Data/processed/df_test.parquet')

In [ ]:
# disable scientific notation for pandas
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df_train

,price,beds,baths,year_of_completion,total_parking_spaces,total_floors,elevators,area_name,city,Latitude,...,type_Villa,type_Villa Compound,furnishing_Unfurnished,completion_status_Ready,year,month,quarter,day_of_week,is_weekend,days_since_posted
0,23000000,4,5,0,0,0,0,5004514.554502,3794105.107071,25.153909,...,True,False,True,True,2023,12,4,1,0,142
1,4794000,5,6,0,0,0,0,3604999.469810,2179499.972323,25.196622,...,False,False,True,False,2024,4,2,4,0,6
2,1650000,3,4,0,0,0,0,1169268.715322,3482826.429041,24.427483,...,False,False,True,True,2024,2,1,5,1,61
3,3300000,4,5,0,0,0,0,3069978.688863,3794105.107071,25.031599,...,False,False,True,True,2024,3,1,4,0,55
4,2350000,2,3,2007,0,53,0,5233059.993865,3794105.107071,25.077802,...,False,False,False,True,2023,11,4,2,0,155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24822,2675000,1,2,2020,729,61,12,3351285.289844,3794105.107071,25.078641,...,False,False,False,True,2024,1,1,2,0,99
24823,2032000,4,4,0,0,0,0,1606577.045732,3794105.107071,24.982883,...,False,False,True,False,2024,4,2,1,0,23
24824,8000000,3,4,0,0,0,0,4941997.576086,3482826.429041,24.479728,...,False,False,False,True,2024,1,1,2,0,106
24825,4200000,3,2,2009,237,15,2,10547845.322011,3794105.107071,25.118088,...,False,False,True,True,2024,4,2,2,0,8


In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24537 entries, 0 to 24827
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   price                         24537 non-null  int64  
 1   beds                          24537 non-null  int64  
 2   baths                         24537 non-null  int64  
 3   year_of_completion            24537 non-null  int64  
 4   total_parking_spaces          24537 non-null  int64  
 5   total_floors                  24537 non-null  int64  
 6   elevators                     24537 non-null  int64  
 7   area_name                     24537 non-null  float64
 8   city                          24537 non-null  float64
 9   Latitude                      24537 non-null  float64
 10  Longitude                     24537 non-null  float64
 11  price_log1p                   24537 non-null  float64
 12  total_building_area_sqft_log  24537 non-null  float64
 13  averag

In [ ]:
df_train.describe()

,price,beds,baths,year_of_completion,total_parking_spaces,total_floors,elevators,area_name,city,Latitude,...,average_rent_log,beds_per_bath,beds_x_baths,floors_x_elevators,year,month,quarter,day_of_week,is_weekend,days_since_posted
count,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,...,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000,24537.000000
mean,3462729.590374,2.166361,2.980275,680.360191,171.719648,13.281982,2.279822,3404396.570673,3462994.627581,25.036572,...,5.748393,0.676361,8.996862,87.333619,2023.859029,4.091657,1.766149,2.392632,0.138036,57.655215
std,7016508.301194,1.547673,1.761733,953.616961,351.202664,20.299082,4.378829,2687494.212410,676389.224829,0.442723,...,5.878334,0.313285,11.557445,214.652828,0.351842,2.771093,0.933128,1.769200,0.344945,64.148290
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,547628.276353,1319248.860526,15.175847,...,0.000000,0.000000,0.000000,0.000000,2022.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,1100000.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,1606577.045732,3482826.429041,25.026710,...,0.000000,0.500000,2.000000,0.000000,2024.000000,3.000000,1.000000,1.000000,0.000000,10.000000
50%,1999999.000000,2.000000,3.000000,0.000000,0.000000,0.000000,0.000000,2823301.604494,3794105.107071,25.078641,...,0.000000,0.666667,6.000000,0.000000,2024.000000,4.000000,2.000000,2.000000,0.000000,38.000000
75%,3500000.000000,3.000000,4.000000,2011.000000,203.000000,22.000000,4.000000,3933416.958308,3794105.107071,25.189427,...,11.688810,1.000000,12.000000,44.000000,2024.000000,4.000000,2.000000,4.000000,0.000000,78.000000
max,482500000.000000,11.000000,11.000000,2024.000000,2929.000000,89.000000,42.000000,24840321.707342,3870096.263294,25.797689,...,13.155732,4.000000,121.000000,2325.000000,2024.000000,12.000000,4.000000,6.000000,1.000000,813.000000


In [ ]:
df_train['price'].describe()

,price
count,24537.000000
mean,3462729.590374
std,7016508.301194
min,0.000000
25%,1100000.000000
50%,1999999.000000
75%,3500000.000000
max,482500000.000000


In [ ]:
X_train = df_train.drop(['price', 'price_log1p'], axis = 1)

y_train = df_train['price_log1p']

In [ ]:
cv3 = KFold(
    n_splits = 3,
    shuffle = True,
    random_state = 123
)

In [ ]:
from sklearn.ensemble import BaggingRegressor

In [ ]:
# # Random Forest = Bagging + DT + feature randomization (strictly better)
# rf_pipeline = Pipeline([
#     ('model', RandomForestRegressor(
#         n_jobs=-1,
#         oob_score=True,
#         random_state=124
#     ))
# ])

# # RandomizedSearchCV samples from distributions — covers far more combinations
# # than GridSearchCV at the same compute cost
# param_dist = {
#     'model__n_estimators':        randint(100, 500),       # more trees = more stable
#     'model__max_depth':           [None, 10, 15, 20, 30],  # None = full depth
#     'model__min_samples_split':   randint(2, 20),
#     'model__min_samples_leaf':    randint(1, 10),
#     'model__max_features':        ['sqrt', 'log2', 0.3, 0.5, 0.7],  # key RF knob
#     'model__max_samples':         uniform(0.6, 0.4),       # bootstrap sample ratio [0.6, 1.0]
#     'model__ccp_alpha':           uniform(0.0, 0.005),     # pruning
#     'model__min_impurity_decrease': uniform(0.0, 0.001),
# }

# rf_random_search = RandomizedSearchCV(
#     estimator=rf_pipeline,
#     param_distributions=param_dist,
#     n_iter=60,              # try 60 random combos — tune up/down for time budget
#     scoring='neg_mean_absolute_error',
#     cv=cv2,
#     n_jobs=-1,
#     verbose=2,
#     random_state=124,
#     refit=True              # auto-refits best model on full X_train
# )

# rf_random_search.fit(X_train, y_train)

# model_rf_best = rf_random_search.best_estimator_

# print("Best params:", rf_random_search.best_params_)
# print("Best CV MAE:", -rf_random_search.best_score_)
# print("OOB score (R²):", model_rf_best.named_steps['model'].oob_score_)

# # Feature importance (bonus insight)
# importances = model_rf_best.named_steps['model'].feature_importances_
# feat_imp = (
#     pd.Series(importances, index=X_train.columns)
#     .sort_values(ascending=False)
#     .head(15)
# )
# print("\nTop 15 features:\n", feat_imp)

In [ ]:
# from sklearn.tree import DecisionTreeRegressor

# bagging_pipeline = Pipeline([
#     ('model', BaggingRegressor(
#         estimator=DecisionTreeRegressor(random_state=124),
#         n_jobs=-1,
#         oob_score=True,
#         random_state=124
#     ))
# ])

# param_dist = {
#     'model__n_estimators':               randint(50, 300),        # wider range, more trees
#     'model__max_samples':                uniform(0.5, 0.5),       # samples ratio [0.5, 1.0]
#     'model__max_features':               uniform(0.5, 0.5),       # feature ratio [0.5, 1.0]
#     'model__bootstrap':                  [True],
#     'model__bootstrap_features':         [True, False],           # also randomize features
#     'model__estimator__max_depth':       [5, 7, 10, 15, None],
#     'model__estimator__min_samples_split': randint(2, 30),
#     'model__estimator__min_samples_leaf':  randint(1, 15),
#     'model__estimator__max_features':    ['sqrt', 'log2', None],  # per-tree feature selection
#     'model__estimator__ccp_alpha':       uniform(0.0, 0.005),
# }

# bagging_random_search = RandomizedSearchCV(
#     estimator=bagging_pipeline,
#     param_distributions=param_dist,
#     n_iter=60,               # 60 random combos — adjust up/down for time budget
#     scoring='neg_mean_absolute_error',
#     cv=cv3,
#     n_jobs=-1,
#     verbose=2,
#     random_state=124,
#     refit=True
# )

# bagging_random_search.fit(X_train, y_train)

# model_bagging_best = bagging_random_search.best_estimator_

# print("Best params:", bagging_random_search.best_params_)
# print("Best CV (negative) MAE:", -bagging_random_search.best_score_)
# print("OOB score (R²):", model_bagging_best.named_steps['model'].oob_score_)

In [ ]:
# joblib.dump(model_bagging_best, '/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_bagging_best-02.pkl')

In [ ]:
# from xgboost import XGBRegressor

# cv3 = KFold(n_splits=3, shuffle=True, random_state=123)

# xgb_pipeline = Pipeline([
#     ('model', XGBRegressor(
#         objective='reg:squarederror',  # regression on log target
#         tree_method='hist',            # fastest for tabular data
#         n_jobs=-1,
#         random_state=124,
#         verbosity=0
#     ))
# ])

# param_dist = {
#     # --- number of trees ---
#     'model__n_estimators':        randint(200, 1000),

#     # --- tree structure ---
#     'model__max_depth':           randint(3, 10),          # shallow trees = less overfit
#     'model__min_child_weight':    randint(1, 20),          # min samples in leaf (XGB style)
#     'model__gamma':               uniform(0.0, 0.5),       # min loss reduction to split

#     # --- sampling (like bagging) ---
#     'model__subsample':           uniform(0.5, 0.5),       # row sampling [0.5, 1.0]
#     'model__colsample_bytree':    uniform(0.4, 0.6),       # feature sampling per tree [0.4, 1.0]
#     'model__colsample_bylevel':   uniform(0.5, 0.5),       # feature sampling per level

#     # --- regularization ---
#     'model__learning_rate':       uniform(0.01, 0.29),     # eta [0.01, 0.30]
#     'model__reg_alpha':           uniform(0.0, 1.0),       # L1 — drives sparse features to 0
#     'model__reg_lambda':          uniform(0.5, 4.5),       # L2 — weight shrinkage [0.5, 5.0]

#     # --- robustness ---
#     'model__max_delta_step':      [0, 1, 5],               # helps with imbalanced price dist
# }

# xgb_random_search = RandomizedSearchCV(
#     estimator=xgb_pipeline,
#     param_distributions=param_dist,
#     n_iter=60,
#     scoring='neg_mean_absolute_error',   # MAE on log1p target
#     cv=cv3,
#     n_jobs=-1,
#     verbose=2,
#     random_state=124,
#     refit=True
# )

# xgb_random_search.fit(X_train, y_train)

# model_xgb_best = xgb_random_search.best_estimator_

# print("Best params:", xgb_random_search.best_params_)
# print("Best CV MAE:", -xgb_random_search.best_score_)

# # CV R² (honest, not in-sample)
# cv_r2 = cross_val_score(model_xgb_best, X_train, y_train, scoring='r2', cv=cv3, n_jobs=-1)
# print("\nCV R² scores:", cv_r2)
# print("Mean CV R²:", round(cv_r2.mean(), 4))
# print("Std CV R²:",  round(cv_r2.std(), 4))

# # feature importance
# importances = model_xgb_best.named_steps['model'].feature_importances_
# feat_imp = (
#     pd.Series(importances, index=X_train.columns)
#     .sort_values(ascending=False)
#     .head(15)
# )
# print("\nTop 15 features:\n", feat_imp)

In [ ]:
# joblib.dump(model_xgb_best, '/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_xgb_best-02.pkl')

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform

cv = KFold(n_splits=3, shuffle=True, random_state=123)

xgb_pipeline = Pipeline([
    ('model', XGBRegressor(
        objective='reg:squarederror',
        tree_method='hist',          # or 'gpu_hist' if GPU available
        grow_policy='lossguide',     # deep, leaf-wise growth (DL-like)
        max_leaves=1024,             # allow very deep trees
        n_jobs=-1,
        random_state=124,
        verbosity=0
    ))
])

param_dist = {
    # --- boosting rounds ---
    'model__n_estimators': randint(500, 2000),

    # --- tree depth / leaf complexity ---
    'model__max_depth': randint(6, 16),          # deeper trees
    'model__max_leaves': randint(64, 1024),      # leaf-wise growth
    'model__min_child_weight': randint(1, 30),

    # --- histogram resolution ---
    'model__max_bin': randint(64, 512),

    # --- sampling ---
    'model__subsample': uniform(0.6, 0.4),       # [0.6, 1.0]
    'model__colsample_bytree': uniform(0.4, 0.6),
    'model__colsample_bylevel': uniform(0.4, 0.6),
    'model__colsample_bynode': uniform(0.4, 0.6),

    # --- regularization ---
    'model__learning_rate': uniform(0.01, 0.25),
    'model__reg_alpha': uniform(0.0, 2.0),       # stronger L1
    'model__reg_lambda': uniform(0.5, 5.0),      # stronger L2

    # --- split sensitivity ---
    'model__gamma': uniform(0.0, 0.5),

    # --- stability for skewed targets ---
    'model__max_delta_step': [0, 1, 5, 10],

    # --- advanced: interaction constraints ---
    'model__interaction_constraints': [
        None,
        [["beds", "baths", "beds_x_baths"]],
        [["Latitude", "Longitude", "area_name"]],
        [["city", "type_Residential Plot", "type_Villa"]],
    ]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist,
    n_iter=80,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=124,
    refit=True
)

xgb_search.fit(X_train, y_train)

model_xgb_best = xgb_search.best_estimator_

print("Best params:", xgb_search.best_params_)
print("Best CV MAE:", -xgb_search.best_score_)

Fitting 3 folds for each of 80 candidates, totalling 240 fits
Best params: {'model__colsample_bylevel': np.float64(0.7574264031321645), 'model__colsample_bynode': np.float64(0.9955643169762546), 'model__colsample_bytree': np.float64(0.662787365394268), 'model__gamma': np.float64(0.007656265181360866), 'model__interaction_constraints': None, 'model__learning_rate': np.float64(0.09995076993251852), 'model__max_bin': 350, 'model__max_delta_step': 0, 'model__max_depth': 13, 'model__max_leaves': 854, 'model__min_child_weight': 19, 'model__n_estimators': 1308, 'model__reg_alpha': np.float64(0.3822543757961414), 'model__reg_lambda': np.float64(4.814704843751463), 'model__subsample': np.float64(0.9243781003052267)}
Best CV MAE: 0.1566094212423463


In [ ]:
# CV R² (honest, not in-sample)
cv_r2 = cross_val_score(model_xgb_best, X_train, y_train, scoring='r2', cv=cv3, n_jobs=-1)
print("\nCV R² scores:", cv_r2)
print("Mean CV R²:", round(cv_r2.mean(), 4))
print("Std CV R²:",  round(cv_r2.std(), 4))

# feature importance
importances = model_xgb_best.named_steps['model'].feature_importances_
feat_imp = (
    pd.Series(importances, index=X_train.columns)
    .sort_values(ascending=False)
    .head(15)
)
print("\nTop 15 features:\n", feat_imp)

In [ ]:
joblib.dump(model_xgb_best, '/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_xgb_gpu.pkl')

['/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_xgb_gpu.pkl']

In [ ]:
import mlflow
import mlflow.xgboost

with mlflow.start_run():
    mlflow.xgboost.log_model(
        xgb_model=model_xgb_best.named_steps['model'],
        artifact_path="model"
    )

2026/05/09 19:14:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [19]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform

cv = KFold(n_splits=3, shuffle=True, random_state=123)

lgbm_pipeline = Pipeline([
    ('model', LGBMRegressor(
        objective='regression',
        boosting_type='gbdt',          # can switch to 'goss' in search
        n_jobs=-1,
        random_state=124,
        verbose=-1
    ))
])

param_dist = {
    # --- boosting rounds ---
    'model__n_estimators': randint(500, 1000),

    # --- tree complexity ---
    'model__max_depth': randint(-1, 16),          # -1 = no limit, up to 16 deep
    'model__num_leaves': randint(64, 2048),       # DL-level capacity
    'model__min_child_samples': randint(5, 100),  # LGBM's main regularizer

    # --- histogram resolution ---
    'model__max_bin': randint(64, 512),

    # --- sampling ---
    'model__subsample': uniform(0.6, 0.4),        # [0.6, 1.0]
    'model__subsample_freq': [1, 3, 5, 7],
    'model__colsample_bytree': uniform(0.4, 0.6),
    'model__colsample_bynode': uniform(0.4, 0.6),

    # --- regularization ---
    'model__learning_rate': uniform(0.01, 0.20),
    'model__reg_alpha': uniform(0.0, 2.0),
    'model__reg_lambda': uniform(0.5, 5.0),
    'model__min_split_gain': uniform(0.0, 1.0),

    # --- advanced boosting ---
    'model__boosting_type': ['gbdt', 'goss'],     # GOSS = deep, aggressive
    'model__bagging_fraction': uniform(0.6, 0.4),
    'model__feature_fraction': uniform(0.4, 0.6),

    # --- monotonic constraints (optional) ---
    'model__monotone_constraints': [
        None,
        [0] * len(X_train.columns),               # no monotonicity
        [1 if "beds" in c or "baths" in c else 0 for c in X_train.columns],
        [-1 if "distance" in c else 0 for c in X_train.columns],
    ]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=124,
    refit=True
)

lgbm_search.fit(X_train, y_train)

model_lgbm_best = lgbm_search.best_estimator_

print("Best params:", lgbm_search.best_params_)
print("Best CV MAE:", -lgbm_search.best_score_)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best params: {'model__bagging_fraction': np.float64(0.67888430632993), 'model__boosting_type': 'gbdt', 'model__colsample_bynode': np.float64(0.9641156972353172), 'model__colsample_bytree': np.float64(0.5475355122763645), 'model__feature_fraction': np.float64(0.9161339609322244), 'model__learning_rate': np.float64(0.017948010597501677), 'model__max_bin': 253, 'model__max_depth': 12, 'model__min_child_samples': 25, 'model__min_split_gain': np.float64(0.39825651046242794), 'model__monotone_constraints': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'model__n_estimators': 568, 'model__num_leaves': 1493, 'model__reg_alpha': np.float64(1.6382324374500163), 'model__reg_lambda': np.float64(4.830784542778685), 'model__subsample': np.float64(0.9863184187923258), 'model__subsample_freq': 3}
Best CV MAE: 0.18320391314714646


In [22]:
joblib.dump(model_lgbm_best, '/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_lgbm_gpu.pkl')

['/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_lgbm_gpu.pkl']

In [27]:
from catboost import CatBoostRegressor

catboost_pipeline = Pipeline([
    ('model', CatBoostRegressor(
        loss_function='RMSE',     # regression on log target
        random_seed=124,
        verbose=0                 # suppress CatBoost logs
    ))
])

param_dist = {
    # --- number of trees ---
    'model__iterations':          randint(200, 1000),      # CatBoost's n_estimators

    # --- tree structure ---
    'model__depth':               randint(3, 10),          # CatBoost's max_depth (max=16)
    'model__min_data_in_leaf':    randint(1, 50),          # min samples in leaf

    # --- sampling (like bagging) ---
    'model__subsample':           uniform(0.5, 0.5),       # row sampling [0.5, 1.0]
    'model__colsample_bylevel':   uniform(0.4, 0.6),       # feature sampling per level

    # --- regularization ---
    'model__learning_rate':       uniform(0.01, 0.29),     # eta [0.01, 0.30]
    'model__l2_leaf_reg':         uniform(1.0, 9.0),       # L2 — CatBoost's reg_lambda [1, 10]
    'model__random_strength':     uniform(0.0, 2.0),       # randomness for scoring splits

    # --- extra CatBoost specific ---
    'model__bagging_temperature': uniform(0.0, 1.0),       # controls Bayesian bootstrap (0=no random, 1=full)
    'model__border_count':        [32, 64, 128, 254],      # histogram bins — like max_bin in LGBM
    'model__grow_policy':         ['SymmetricTree', 'Depthwise', 'Lossguide'],  # tree growth strategy
}

catboost_random_search = RandomizedSearchCV(
    estimator=catboost_pipeline,
    param_distributions=param_dist,
    n_iter=60,
    scoring='neg_mean_absolute_error',
    cv=cv3,
    n_jobs=-1,
    verbose=2,
    random_state=124,
    refit=True
)

catboost_random_search.fit(X_train, y_train)

model_catboost_best = catboost_random_search.best_estimator_

# print("Best params:", catboost_random_search.best_params_)
# print("Best CV MAE:", -catboost_random_search.best_score_)

# # CV R²
# cv_r2 = cross_val_score(model_catboost_best, X_train, y_train, scoring='r2', cv=cv3, n_jobs=-1)
# print("\nCV R² scores:", cv_r2)
# print("Mean CV R²:", round(cv_r2.mean(), 4))
# print("Std CV R²:",  round(cv_r2.std(), 4))

# # feature importance
# importances = model_catboost_best.named_steps['model'].feature_importances_
# feat_imp = (
#     pd.Series(importances, index=X_train.columns)
#     .sort_values(ascending=False)
#     .head(15)
# )
# print("\nTop 15 features:\n", feat_imp)

Fitting 3 folds for each of 60 candidates, totalling 180 fits


KeyboardInterrupt: 

In [ ]:
print("Best params:", catboost_random_search.best_params_)
print("Best CV MAE:", -catboost_random_search.best_score_)

# CV R² — use best_index_ from search results instead of cross_val_score
cv_r2 = catboost_random_search.cv_results_['mean_test_score'][catboost_random_search.best_index_]
print("\nBest CV MAE (from results):", -cv_r2)

# R² manually from in-sample prediction (no cloning needed)
from sklearn.metrics import r2_score
y_pred_train = model_catboost_best.predict(X_train)
print("R² (train, log scale):", round(r2_score(y_train, y_pred_train), 4))

# feature importance
importances = model_catboost_best.named_steps['model'].feature_importances_
feat_imp = (
    pd.Series(importances, index=X_train.columns)
    .sort_values(ascending=False)
    .head(15)
)
print("\nTop 15 features:\n", feat_imp)

Best params: {'model__bagging_temperature': np.float64(0.019450456814706696), 'model__border_count': 254, 'model__colsample_bylevel': np.float64(0.4047847150282479), 'model__depth': 9, 'model__grow_policy': 'Depthwise', 'model__iterations': 343, 'model__l2_leaf_reg': np.float64(9.571390929333507), 'model__learning_rate': np.float64(0.19133942898395673), 'model__min_data_in_leaf': 2, 'model__random_strength': np.float64(0.5803887953845652), 'model__subsample': np.float64(0.7484570873123457)}
Best CV MAE: 0.15585633106165042

Best CV MAE (from results): 0.15585633106165042
R² (train, log scale): 0.9725

Top 15 features:
 beds_x_baths               30.148853
area_name                  28.888892
Latitude                    8.019742
beds                        6.113649
average_rent_log            4.273387
Longitude                   3.768378
baths                       3.758611
beds_per_bath               3.175582
city                        2.824825
type_Residential Plot       1.102107
ele

In [ ]:
joblib.dump(model_catboost_best, '/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_catboost_best-02.pkl')

['/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_catboost_best-02.pkl']

In [ ]:
from catboost import CatBoostRegressor

cat_params = model_catboost_best.named_steps['model'].get_params()

# convert numpy types → python types
cat_params = {k: float(v) if isinstance(v, (np.floating,)) else v
              for k, v in cat_params.items()}

model_catboost_clean = CatBoostRegressor(**cat_params)

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge  # better meta-learner than LinearRegression
from sklearn.model_selection import GridSearchCV

# stacking pipeline using 3 best models
stack_pipeline = Pipeline([
    ('model', StackingRegressor(
        estimators=[
            ('xgb',      model_xgb_best.named_steps['model']),
            ('lgbm',     model_lgbm_best.named_steps['model']),
            ('catboost', model_catboost_clean),
            ('bagging',  model_bagging_best.named_steps['model']),
        ],
        final_estimator=Ridge(),   # Ridge >> LinearRegression as meta-learner (handles correlated predictions)
        cv=5,                      # inner CV for generating meta-features
        n_jobs=-1,
        passthrough=False          # only pass base model predictions to meta-learner
    ))
])

# small grid — stacking is very heavy, keep it minimal
param_grid = {
    'model__final_estimator__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],  # Ridge regularization
}

stack_grid_search = GridSearchCV(
    estimator=stack_pipeline,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv3,
    n_jobs=-1,
    verbose=1
)

stack_grid_search.fit(X_train, y_train)

model_stacking_best = stack_grid_search.best_estimator_

print("Best params:", stack_grid_search.best_params_)
print("Best CV MAE:", -stack_grid_search.best_score_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best params: {'model__final_estimator__alpha': 1.0}
Best CV MAE: 0.15326243374878532


In [ ]:
joblib.dump(model_stacking_best, '/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_stacking_best-02.pkl')

['/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/model_stacking_best-02.pkl']